# Harmonização LULC

Aplicação da harmonização das classes de uso e ocupação do solo.

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

## Selecção do ano de processamento

In [2]:
# Escolher o Ano
year = 2018

### Definição dos caminhos

In [3]:
fld = "/code/data/raw/lulc"
#aoi_file = "/code/data/processed/pnse/aoi/pnse.shp"
aoi_file = "/code/data/processed/centro/aoi/centro.shp"
harm_file ="/code/data/processed/centro/lulc/harmonized_tables/tabela_harmonizacao.csv"

out_clip = "/code/data/processed/centro/lulc/clip_aoi"
out_harm = "/code/data/processed/centro/lulc/harmonized_vectors"

Path(out_clip).mkdir(parents=True, exist_ok=True)
Path(out_harm).mkdir(parents=True, exist_ok=True)

In [4]:
cos = {
    1995: {
        "file": f"{fld}/COS1995v2-S1.gpkg",
        "layer": "COS1995v2",
        "n1": "COS95n1_C",
        "code": "COS95n4_C",
        "label": "COS95n4_L",
    },
    2007: {
        "file": f"{fld}/COS2007v3-S1.gpkg",
        "layer": "COS2007v3",
        "n1": "COS07n1_C",
        "code": "COS07n4_C",
        "label": "COS07n4_L",
    },
    2010: {
        "file": f"{fld}/COS2010v2-S1.gpkg",
        "layer": "COS2010v2",
        "n1": "COS10n1_C",
        "code": "COS10n4_C",
        "label": "COS10n4_L",
    },
    2015: {
        "file": f"{fld}/COS2015v2-S1.gpkg",
        "layer": "COS2015v2",
        "n1": "COS15n1_C",
        "code": "COS15n4_C",
        "label": "COS15n4_L",
    },
    2018: {
        "file": f"{fld}/COS2018v2-S1.gpkg",
        "layer": "COS2018v2",
        "n1": "COS18n1_C",
        "code": "COS18n4_C",
        "label": "COS18n4_L",
    },
}

In [5]:
#ler aoi e tabela de harmonização
aoi = gpd.read_file(aoi_file)

harm = pd.read_csv(
    harm_file,
    dtype={
        "year": int,
        "code_original": str,
        "label_original": str,
        "label_harmonizada": str,
        "incluir_modelo": int,
        "obs": str,
        "id_harm": "Int64",
    },
)

In [6]:
#funções auxiliares de normalização
def norm_code(s):
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def norm_label(s):
    return (
        s.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [7]:
#configuração do ano selecionado
cfg = cos[year]

clip_file = f"{out_clip}/cos_{year}_clip.gpkg"
clip_layer = f"cos_{year}_clip"

harm_out = f"{out_harm}/cos_{year}_harm.gpkg"
harm_layer = f"cos_{year}_harm"

drop_report = f"{out_harm}/cos_{year}_classes_excluidas.csv"

In [8]:
sample = gpd.read_file(cfg["file"], layer=cfg["layer"], rows=1)

aoi_year = aoi.copy()
if aoi_year.crs != sample.crs:
    aoi_year = aoi_year.to_crs(sample.crs)

aoi_year = aoi_year.dissolve()
aoi_bbox = tuple(aoi_year.total_bounds)

In [9]:
#ler cos
cos_bbox = gpd.read_file(
    cfg["file"],
    layer=cfg["layer"],
    bbox=aoi_bbox,
)

print(f"Ano: {year}")
print(f"Feições COS lidas por bbox: {len(cos_bbox)}")

Ano: 2018
Feições COS lidas por bbox: 261347


### Clip à AOI e exclusão de classes 

In [10]:
import pyogrio

aoi_geom = aoi_year.geometry.iloc[0]

cos_clip = pyogrio.read_dataframe(
    cfg["file"],
    layer=cfg["layer"],
    mask=aoi_geom,
    where=f'CAST("{cfg["n1"]}" AS TEXT) NOT IN (\'1\', \'9\')',
    use_arrow=True,
)

cos_clip = cos_clip.loc[
    cos_clip.geometry.notna() & ~cos_clip.geometry.is_empty
].copy()

print(f"Ano: {year}")
print(f"Feições COS lidas com filtro espacial: {len(cos_clip)}")

Ano: 2018
Feições COS lidas com filtro espacial: 188416


In [11]:
"""cos_clip = gpd.clip(cos_bbox, aoi_year)
cos_clip = cos_clip.loc[
    cos_clip.geometry.notna() & ~cos_clip.geometry.is_empty
].copy()

print(f"Feições COS após clip exacto: {len(cos_clip)}")"""

'cos_clip = gpd.clip(cos_bbox, aoi_year)\ncos_clip = cos_clip.loc[\n    cos_clip.geometry.notna() & ~cos_clip.geometry.is_empty\n].copy()\n\nprint(f"Feições COS após clip exacto: {len(cos_clip)}")'

In [12]:
"""cos_clip = cos_clip[
    ~cos_clip[cfg["n1"]].astype(str).isin(["1", "9"])
].copy()

print(f"Feições após exclusão de n1=1 e n1=9: {len(cos_clip)}")"""

'cos_clip = cos_clip[\n    ~cos_clip[cfg["n1"]].astype(str).isin(["1", "9"])\n].copy()\n\nprint(f"Feições após exclusão de n1=1 e n1=9: {len(cos_clip)}")'

In [13]:
"""cos_clip.to_file(clip_file, layer=clip_layer, driver="GPKG")

print(f"Recorte guardado em: {clip_file}")"""

'cos_clip.to_file(clip_file, layer=clip_layer, driver="GPKG")\n\nprint(f"Recorte guardado em: {clip_file}")'

## Preparação da tabela de harmonização para o ano selecionado


In [14]:
harm_year = harm.loc[harm["year"] == year].copy()

harm_year["code_original"] = norm_code(harm_year["code_original"])
harm_year["label_original"] = norm_label(harm_year["label_original"])

cos_clip[cfg["code"]] = norm_code(cos_clip[cfg["code"]])
cos_clip[cfg["label"]] = norm_label(cos_clip[cfg["label"]])

harm_year = harm_year.drop_duplicates(
    subset=["code_original", "label_original"]
).copy()

print(f"Linhas da tabela de harmonização para {year}: {len(harm_year)}")

Linhas da tabela de harmonização para 2018: 33


In [15]:
#identificar classes da cos sem correspondência no csv
classes_cos = (
    cos_clip[[cfg["code"], cfg["label"]]]
    .drop_duplicates()
    .rename(
        columns={
            cfg["code"]: "code_original",
            cfg["label"]: "label_original",
        }
    )
)

classes_excluidas = classes_cos.merge(
    harm_year[["code_original", "label_original"]],
    on=["code_original", "label_original"],
    how="left",
    indicator=True,
)

classes_excluidas = classes_excluidas.loc[
    classes_excluidas["_merge"] == "left_only",
    ["code_original", "label_original"],
].sort_values(["code_original", "label_original"])

classes_excluidas.to_csv(drop_report, index=False)

print(f"Classes da COS sem correspondência no csv: {len(classes_excluidas)}")
print(f"Relatório guardado em: {drop_report}")

classes_excluidas.head(20)

Classes da COS sem correspondência no csv: 5
Relatório guardado em: /code/data/processed/centro/lulc/harmonized_vectors/cos_2018_classes_excluidas.csv


,code_original,label_original
31,7.1.1.1,"Praias, dunas e areais interiores"
32,7.1.1.2,"Praias, dunas e areais costeiros"
35,8.1.1.1,Pauis
36,8.1.2.1,Sapais
37,8.1.2.2,Zonas entremarés


In [16]:
#harmonizar e filtrar só classes que entram no modelo
cos_harm = cos_clip.merge(
    harm_year,
    left_on=[cfg["code"], cfg["label"]],
    right_on=["code_original", "label_original"],
    how="inner",
    validate="many_to_one",
)

print(f"Feições com correspondência no csv: {len(cos_harm)}")

cos_harm = cos_harm.loc[cos_harm["incluir_modelo"] == 1].copy()

cos_harm = cos_harm.rename(
    columns={
        "label_harmonizada": "classe_harm",
    }
)

keep_cols = list(cos_clip.columns) + [
    "id_harm",
    "classe_harm",
    "incluir_modelo",
    "obs",
]

cos_harm = cos_harm[keep_cols].copy()

print(f"Feições finais após filtro incluir_modelo == 1: {len(cos_harm)}")

Feições com correspondência no csv: 188158
Feições finais após filtro incluir_modelo == 1: 188158


In [17]:
#guardar cos harmonizada
cos_harm.to_file(harm_out, layer=harm_layer, driver="GPKG")

print(f"COS harmonizada guardada em: {harm_out}")

COS harmonizada guardada em: /code/data/processed/centro/lulc/harmonized_vectors/cos_2018_harm.gpkg


In [18]:
#verificação

resumo = (
    cos_harm[["id_harm", "classe_harm"]]
    .value_counts()
    .rename("n")
    .reset_index()
    .sort_values(["id_harm", "classe_harm"])
)

print(f"Número de classes harmonizadas finais: {len(resumo)}")
resumo.head(50)

print("Resumo da execução")
print("-" * 40)
print(f"Ano: {year}")
print(f"Feições recortadas: {len(cos_clip)}")
print(f"Classes excluídas por não existirem no csv: {len(classes_excluidas)}")
print(f"Feições finais harmonizadas: {len(cos_harm)}")
print(f"GPKG recortado: {clip_file}")
print(f"GPKG harmonizado: {harm_out}")
print(f"CSV classes excluídas: {drop_report}")

Número de classes harmonizadas finais: 32
Resumo da execução
----------------------------------------
Ano: 2018
Feições recortadas: 188416
Classes excluídas por não existirem no csv: 5
Feições finais harmonizadas: 188158
GPKG recortado: /code/data/processed/centro/lulc/clip_aoi/cos_2018_clip.gpkg
GPKG harmonizado: /code/data/processed/centro/lulc/harmonized_vectors/cos_2018_harm.gpkg
CSV classes excluídas: /code/data/processed/centro/lulc/harmonized_vectors/cos_2018_classes_excluidas.csv


In [19]:
classes_excluidas

,code_original,label_original
31,7.1.1.1,"Praias, dunas e areais interiores"
32,7.1.1.2,"Praias, dunas e areais costeiros"
35,8.1.1.1,Pauis
36,8.1.2.1,Sapais
37,8.1.2.2,Zonas entremarés
